# AIC 2026 — Summary từng video bằng Qwen3.5-4B (2 GPU Kaggle)

`Qwen/Qwen3.5-4B` là **LLM text-only**, không nhìn được video. Notebook này vì vậy không đọc file `.mp4`
mà tóm tắt từ *bằng chứng dạng text* đã có sẵn trong repo, gộp theo timeline của từng video:

| Nguồn | Thư mục | Ngôn ngữ |
|---|---|---|
| Caption keyframe (VLM/Florence-2) | `Feature_Dataset/Image_captioning` | English |
| Transcript (PhoWhisper) | `Feature_Dataset/Transcript_Extract` | Vietnamese |

Mỗi video ra **một summary tiếng Anh** (`SUMMARY` / `TOPICS` / `ENTITIES`) ghi vào `Summary_video/<video_id>.json`.
Phần lời nói tiếng Việt được model dịch nghĩa sang English ngay trong lúc tóm tắt, nên output đồng nhất một ngôn ngữ
để đưa vào index truy hồi.

**Chạy 2 GPU**: mỗi GPU giữ một bản model riêng và xử lý một video tại một thời điểm (giống notebook STT).
Video dài bị cắt thành nhiều chunk → tóm tắt từng chunk (map, chạy theo batch trên cùng GPU) → gộp lại thành
summary cuối (reduce).

**Kaggle**: Settings → Accelerator → **GPU T4 x2**, **Internet: On** (tải model từ Hugging Face).
Add Data cho cả dataset caption và dataset transcript. `/kaggle/working` bị xoá khi session kết thúc → nhớ
**Save Version**, rồi lần sau *Add Data → Notebook Output* để resume.

**Colab**: Runtime → GPU (chỉ 1 GPU, notebook tự lùi về chế độ single-GPU).

In [ ]:
import os
import subprocess
import sys

def detect_env():
    """Phát hiện môi trường: 'kaggle' | 'colab' | 'local'.

    Kiểm tra Kaggle TRƯỚC vì /kaggle là dấu hiệu chắc chắn; image của Kaggle có
    thể khiến các tín hiệu của Colab khớp sai.
    """
    if os.path.isdir('/kaggle/input') or os.path.isdir('/kaggle/working') or os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        return 'kaggle'
    # os.name để tránh nhận nhầm khi chạy thử trên Windows ('/content' bị hiểu là C:\content).
    if os.environ.get('COLAB_RELEASE_TAG') or 'google.colab' in sys.modules or (os.name == 'posix' and os.path.isdir('/content')):
        return 'colab'
    return 'local'

ENV = detect_env()
print('Môi trường:', ENV)

subprocess.run(['nvidia-smi'], check=False)

# Qwen3.5 cần transformers mới; nếu bản pip release chưa nhận ra kiến trúc này thì
# cài từ nhánh main: pip install -U git+https://github.com/huggingface/transformers
# KHÔNG đụng vào torch/numpy của Kaggle/Colab:
# transformers không phụ thuộc torch nên nâng cấp an toàn, accelerate chỉ yêu cầu torch>=2.0.
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-U', 'transformers', 'accelerate'], check=True)

import transformers
print('transformers:', transformers.__version__)

In [ ]:
# Chỉ Colab cần mount Drive. Trên Kaggle dữ liệu đã có sẵn ở /kaggle/input.
if ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Bỏ qua mount Drive (ENV =', ENV, ')')

## Cấu hình

Notebook tự đi tìm thư mục caption và transcript trong `/kaggle/input` (tên dataset trên Kaggle thường bị bọc
thêm một hai cấp, nên dò theo **tên thư mục** thay vì hard-code đường dẫn). Nếu dò sai thì điền tay vào
`CAPTION_DIRS_OVERRIDE` / `TRANSCRIPT_DIRS_OVERRIDE` — cell sẽ in ra những gì nó tìm được để đối chiếu.

Transcript lấy từ `/kaggle/input/datasets/kitnehi1211/transcript/Transcript_Extract` (đã điền sẵn vào
`TRANSCRIPT_DIRS_OVERRIDE`). Nếu chưa Add dataset caption thì notebook vẫn chạy được nhưng summary chỉ
dựa trên lời nói — cell sẽ in `LƯU Ý` để bạn biết.

**Chọn folder chạy**: `TARGET_FOLDERS` liệt kê sẵn cả 14 folder của dataset transcript kèm số video —
comment / bỏ comment để chọn đợt. Mặc định là đợt 1 (7 folder `L21_a` → `L26_b`, 415 video); đợt 2 thì
comment nhóm trên, bỏ comment nhóm dưới (7 folder `L26_c` → `L30_a`, 458 video). Muốn chạy hết thì để
`TARGET_FOLDERS = ['.']`.

Cell **Quét nguồn** in checklist `[x]/[ ]` của cả 14 folder kèm số video để đối chiếu trước khi chạy cell
tốn GPU, và báo lỗi ngay nếu có tên folder không tồn tại. Output mỗi video một file `.json` riêng nên các
lượt ghi cùng thư mục mà không đè nhau; folder đã xong để lại trong danh sách cũng chỉ bị `SKIP`.

In [ ]:
from pathlib import Path

try:
    ENV
except NameError:
    ENV = ('kaggle' if os.path.isdir('/kaggle/input')
           else 'colab' if (os.name == 'posix' and os.path.isdir('/content')) else 'local')

# --- Tên thư mục nguồn (dò theo tên, không hard-code cả đường dẫn) ----------
CAPTION_DIR_NAMES = ['Image_captioning', 'ImageCaptioning', 'Captions', 'VLM_Qwen3.5-2b']
TRANSCRIPT_DIR_NAMES = ['Transcript_Extract', 'Transcripts']

# Điền tay khi biết chắc đường dẫn (nhanh và chắc hơn dò tự động).
CAPTION_DIRS_OVERRIDE = []      # ví dụ: [Path('/kaggle/input/datasets/<user>/<ds>/Image_captioning')]
TRANSCRIPT_DIRS_OVERRIDE = [
    Path('/kaggle/input/datasets/kitnehi1211/transcript/Transcript_Extract'),
]

if ENV == 'kaggle':
    SEARCH_ROOTS = [Path('/kaggle/input'), Path('/kaggle/working')]
    OUTPUT_ROOT = Path('/kaggle/working/Summary_video')
elif ENV == 'colab':
    SEARCH_ROOTS = [Path('/content/drive/MyDrive/AI Challenge')]
    OUTPUT_ROOT = Path('/content/drive/MyDrive/AI Challenge/Summary_video')
else:
    SEARCH_ROOTS = [Path('./Feature_Dataset'), Path('../Feature_Dataset'), Path('../../Feature_Dataset')]
    OUTPUT_ROOT = Path('./Feature_Dataset/Summary_video')

def find_dirs(names, roots, max_depth=4):
    """Tìm mọi thư mục có tên nằm trong `names`, quét tối đa `max_depth` cấp."""
    wanted = {name.lower() for name in names}
    found = []
    for root in roots:
        if not root.is_dir():
            continue
        for depth in range(1, max_depth + 1):
            for path in root.glob('/'.join(['*'] * depth)):
                if path.is_dir() and path.name.lower() in wanted:
                    found.append(path.resolve())
    # Bỏ thư mục nằm trong OUTPUT_ROOT để không tự đọc lại kết quả của chính mình.
    output_resolved = OUTPUT_ROOT.resolve()
    unique = []
    for path in found:
        if path == output_resolved or output_resolved in path.parents:
            continue
        if path not in unique:
            unique.append(path)
    return unique

def resolve_sources(overrides, names):
    """Ưu tiên đường dẫn điền tay; thư mục nào không tồn tại thì bỏ và dò tự động."""
    chosen = [Path(path).resolve() for path in overrides if Path(path).is_dir()]
    missing = [str(path) for path in overrides if not Path(path).is_dir()]
    for path in missing:
        print('  (override không tồn tại, bỏ qua):', path)
    return chosen or find_dirs(names, SEARCH_ROOTS)

CAPTION_DIRS = resolve_sources(CAPTION_DIRS_OVERRIDE, CAPTION_DIR_NAMES)
TRANSCRIPT_DIRS = resolve_sources(TRANSCRIPT_DIRS_OVERRIDE, TRANSCRIPT_DIR_NAMES)

print('Thư mục caption:')
for path in CAPTION_DIRS or ['(không tìm thấy)']:
    print('  -', path)
print('Thư mục transcript:')
for path in TRANSCRIPT_DIRS or ['(không tìm thấy)']:
    print('  -', path)
if not CAPTION_DIRS and not TRANSCRIPT_DIRS and ENV == 'kaggle':
    print('\nCó trong /kaggle/input:', ', '.join(sorted(p.name for p in Path('/kaggle/input').iterdir())))
assert CAPTION_DIRS or TRANSCRIPT_DIRS, (
    'Không tìm thấy nguồn nào. Add Data dataset caption/transcript rồi điền CAPTION_DIRS_OVERRIDE.'
)
if not CAPTION_DIRS:
    print('LƯU Ý: không có caption -> summary chỉ dựa trên lời nói, mất hết thông tin hình ảnh.')
if not TRANSCRIPT_DIRS:
    print('LƯU Ý: không có transcript -> summary chỉ dựa trên hình ảnh, mất hết thông tin lời nói.')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT = OUTPUT_ROOT.resolve()
print('Output:', OUTPUT_ROOT)

# --- Chọn folder chạy ------------------------------------------------------
# 14 folder của dataset transcript, kèm số video. Chọn đợt chạy bằng cách
# comment / bỏ comment: session Kaggle ~9-12h nên mỗi lần 7 folder là vừa.
# Mỗi video ghi ra một file .json riêng nên các lần chạy không đè nhau, và
# folder đã xong ở lần trước có để lại trong danh sách cũng chỉ bị SKIP.
# Để TARGET_FOLDERS = ['.'] nếu muốn chạy hết mọi folder tìm được.
TARGET_FOLDERS = [
    # --- đợt 1: 415 video ---
    'Videos_L21_a',   #  29
    'Videos_L22_a',   #  31
    'Videos_L23_a',   #  25
    'Videos_L24_a',   #  43
    'Videos_L25_a',   #  88
    'Videos_L26_a',   #  99
    'Videos_L26_b',   # 100
    # --- đợt 2: 458 video — comment nhóm trên, bỏ comment nhóm dưới ---
    # 'Videos_L26_c',   # 100
    # 'Videos_L26_d',   # 100
    # 'Videos_L26_e',   #  99
    # 'Videos_L27_a',   #  16
    # 'Videos_L28_a',   #  24
    # 'Videos_L29_a',   #  23
    # 'Videos_L30_a',   #  96
]

VIDEO_PREFIXES = []     # lọc thêm theo prefix nếu cần, ví dụ ['L21', 'L22']; [] = tất cả
RUN_SLICE = ''          # cắt nhỏ tiếp trong lượt, ví dụ '0:20' trên danh sách CHƯA xong
OVERWRITE = False

# --- Model -----------------------------------------------------------------
MODEL_ID = 'Qwen/Qwen3.5-4B'
DTYPE = 'auto'          # 'auto' (bfloat16 nếu GPU hỗ trợ, ngược lại float16) | 'float16' | 'bfloat16'
ENABLE_THINKING = False  # Qwen3.5 thinking mode: đúng hơn chút nhưng tốn 3-5x token -> tắt

# --- Chunk + sinh text -----------------------------------------------------
# Ngân sách token cho phần EVIDENCE của mỗi chunk (không tính prompt/hệ thống).
# T4 16GB với 4B fp16 (~8GB weight) chịu được ~3k token/chunk ở batch 4.
CHUNK_TOKEN_BUDGET = 2600
MAP_BATCH_SIZE = 4       # số chunk sinh song song trên cùng GPU; OOM thì tự lùi về 1
MAX_CHUNKS_PER_VIDEO = 0  # 0 = không giới hạn
MAX_NEW_TOKENS_MAP = 320
MAX_NEW_TOKENS_FINAL = 420

# Họ Qwen3 khuyến nghị KHÔNG dùng greedy (dễ lặp vô tận). Dùng tham số sampling của
# chế độ non-thinking + seed cố định để chạy lại vẫn ra kết quả như cũ.
DO_SAMPLE = True
TEMPERATURE = 0.7
TOP_P = 0.8
TOP_K = 20
SEED = 1234

# --- Nội dung nguồn --------------------------------------------------------
SKIP_DUPLICATE_CAPTIONS = True   # caption của keyframe trùng (đã có duplicate_of) thì bỏ
SPEECH_LANGUAGE = 'Vietnamese'   # ngôn ngữ transcript, dùng trong prompt

## Prompt

Summary viết cho **index truy hồi**, không phải cho người đọc: ép model nói ra thực thể, địa điểm, con số,
hành động — những thứ query của ban tổ chức thường nhắm vào — và cấm nói vòng vo kiểu
"the video shows...". Ba nhãn `SUMMARY` / `TOPICS` / `ENTITIES` được parse bằng regex ở cell sau,
model nào lỡ trả về dạng khác thì toàn bộ text rơi vào `summary` (không mất dữ liệu).

In [ ]:
SYSTEM_PROMPT = (
    'You are an expert video analyst building English metadata for a video retrieval system. '
    'You never see the video itself; you receive an evidence timeline extracted from it: '
    f'VISUAL lines are English captions of keyframes, SPEECH lines are {SPEECH_LANGUAGE} '
    'automatic transcripts. Translate any non-English content into English. '
    'Ground every statement in the evidence; never invent names, numbers or places. '
    'Automatic transcripts are noisy - ignore fragments that make no sense instead of guessing.'
)

MAP_PROMPT = '''Below is part of the evidence timeline of video {video_id} (part {part}/{total}, {start}-{end}).

{evidence}

Write a dense factual English digest of this part, as 3-6 bullet lines starting with "- ".
Each bullet: what is on screen and what is being said, with the approximate timestamp in [mm:ss].
Keep every concrete detail: people and their roles, clothing, objects, counts, locations, organisations,
on-screen text, actions and events. No preamble, no conclusion, bullets only.'''

FINAL_PROMPT = '''Below is the evidence about video {video_id}.

{evidence}

Write English retrieval metadata for this video, in exactly this format:

SUMMARY: one paragraph of 120-200 words describing what happens in the video in chronological order.
State the subject matter, setting, the people involved and the main events. Write plain declarative
sentences about the content itself - do not start with "The video shows" or similar meta phrasing.
TOPICS: 5-10 short topical keywords or phrases, separated by semicolons.
ENTITIES: the named people, organisations, places, dates and numbers that appear, separated by
semicolons; write "none" if the evidence contains none.'''

## Quét nguồn, gộp timeline

Cell này in checklist folder theo `TARGET_FOLDERS`, rồi gộp evidence của các video trong những folder đã chọn.

Mỗi video được gộp thành một chuỗi sự kiện `[mm:ss] VISUAL: ...` / `[mm:ss] SPEECH: ...` sắp theo thời gian.
Caption trùng lặp và caption giống hệt cái liền trước bị loại — chúng chỉ làm phình prompt mà không thêm
thông tin. Video nào không có cả caption lẫn transcript thì bị loại khỏi danh sách ngay ở đây.

In [ ]:
import json
import re

VIDEO_ID_PATTERN = re.compile(r'^L\d{2}_V\d{3}$')

def index_sources(dirs):
    """video_id -> đường dẫn json. File ở thư mục đứng trước thắng (ưu tiên override/dataset chính)."""
    mapping = {}
    for directory in dirs:
        for path in sorted(directory.rglob('*.json')):
            if path.name.startswith('_') or path.stem.endswith('.partial'):
                continue
            if VIDEO_ID_PATTERN.match(path.stem):
                mapping.setdefault(path.stem, path)
    return mapping

CAPTION_FILES = index_sources(CAPTION_DIRS)
TRANSCRIPT_FILES = index_sources(TRANSCRIPT_DIRS)
print(f'Caption: {len(CAPTION_FILES)} video | Transcript: {len(TRANSCRIPT_FILES)} video')

def read_json(path):
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception as error:
        print(f'Bỏ qua file lỗi {path}: {error!r}')
        return None

def caption_events(video_id):
    path = CAPTION_FILES.get(video_id)
    payload = read_json(path) if path else None
    if not payload:
        return []
    events, previous = [], None
    for item in payload.get('keyframes', []):
        if SKIP_DUPLICATE_CAPTIONS and item.get('duplicate_of'):
            continue
        text = (item.get('caption') or '').strip()
        if not text:
            continue
        normalised = re.sub(r'\W+', ' ', text.lower()).strip()
        if normalised == previous:   # caption y hệt cái liền trước -> không thêm thông tin
            continue
        previous = normalised
        try:
            time = float(item.get('pts_time') or 0.0)
        except (TypeError, ValueError):
            time = 0.0
        events.append({'t': time, 'kind': 'VISUAL', 'text': text})
    return events

def speech_events(video_id):
    path = TRANSCRIPT_FILES.get(video_id)
    payload = read_json(path) if path else None
    if not payload:
        return []
    events = []
    for segment in payload.get('segments', []):
        text = (segment.get('text') or '').strip()
        if not text:
            continue
        try:
            time = float(segment.get('video_start') if segment.get('video_start') is not None else segment.get('start') or 0.0)
        except (TypeError, ValueError):
            time = 0.0
        events.append({'t': time, 'kind': 'SPEECH', 'text': text})
    if not events:
        whole = (payload.get('text') or '').strip()
        if whole:
            events.append({'t': 0.0, 'kind': 'SPEECH', 'text': whole})
    return events

def clock(seconds):
    total = max(0, int(round(float(seconds))))
    return f'{total // 60:02d}:{total % 60:02d}'

def build_timeline(video_id):
    """Gộp caption + transcript thành một danh sách sự kiện theo thời gian."""
    visual, speech = caption_events(video_id), speech_events(video_id)
    events = sorted(visual + speech, key=lambda item: (item['t'], item['kind']))
    return {
        'video_id': video_id,
        'events': events,
        'visual_count': len(visual),
        'speech_count': len(speech),
        'has_caption': video_id in CAPTION_FILES,
        'has_transcript': video_id in TRANSCRIPT_FILES,
        'duration_hint': max((item['t'] for item in events), default=0.0),
    }

def event_line(event):
    return f"[{clock(event['t'])}] {event['kind']}: {event['text']}"

def source_folder(path, roots):
    """Tên thư mục video chứa file này, ví dụ 'Videos_L21_a'. '.' nếu nằm ngay gốc."""
    for root in roots:
        if root == path.parent or root in path.parents:
            parts = path.relative_to(root).parts
            return parts[0] if len(parts) > 1 else '.'
    return '.'

# Folder của mỗi video lấy từ phía transcript (thư mục caption phẳng, không có folder).
VIDEO_FOLDER = {video_id: source_folder(path, TRANSCRIPT_DIRS)
                for video_id, path in TRANSCRIPT_FILES.items()}
ALL_FOLDERS = sorted(set(VIDEO_FOLDER.values()))
folder_sizes = {folder: sum(1 for value in VIDEO_FOLDER.values() if value == folder)
                for folder in ALL_FOLDERS}

assert TARGET_FOLDERS, 'TARGET_FOLDERS trống — bỏ comment ít nhất một folder'
run_all_folders = list(TARGET_FOLDERS) == ['.']
selected_folders = ALL_FOLDERS if run_all_folders else list(TARGET_FOLDERS)

unknown = [folder for folder in selected_folders if folder not in folder_sizes]
assert not unknown, (
    f'Folder không có trong dataset: {unknown}\n'
    f'Folder thực có ({len(ALL_FOLDERS)}): {ALL_FOLDERS}'
)

print(f'Dataset có {len(ALL_FOLDERS)} folder; lượt này chạy {len(selected_folders)}:')
for folder in ALL_FOLDERS:
    chosen = folder in set(selected_folders)
    print(f"  {'[x]' if chosen else '[ ]'} {folder:16} {folder_sizes[folder]:4} video")
print(f'  -> tổng lượt này: {sum(folder_sizes[folder] for folder in selected_folders)} video')

wanted_folders = set(selected_folders)
all_video_ids = sorted(video_id for video_id, folder in VIDEO_FOLDER.items() if folder in wanted_folders)

# Video chỉ có caption mà không có transcript thì không thuộc folder nào -> ngoài mọi đợt.
caption_only = sorted(set(CAPTION_FILES) - set(TRANSCRIPT_FILES))
if caption_only:
    print(f'{len(caption_only)} video chỉ có caption, không có transcript -> không nằm trong đợt nào')

if VIDEO_PREFIXES:
    prefixes = tuple(VIDEO_PREFIXES)
    all_video_ids = [video_id for video_id in all_video_ids if video_id.startswith(prefixes)]

timelines = {}
empty_videos = []
for video_id in all_video_ids:
    timeline = build_timeline(video_id)
    if timeline['events']:
        timelines[video_id] = timeline
    else:
        empty_videos.append(video_id)

video_ids = list(timelines)
both = sum(1 for t in timelines.values() if t['has_caption'] and t['has_transcript'])
print(f'{len(video_ids)} video có dữ liệu ({both} video có cả caption và transcript)'
      + (f' | {len(empty_videos)} video rỗng bị loại' if empty_videos else ''))
for video_id in video_ids[:5]:
    timeline = timelines[video_id]
    print(f"  - {video_id}: {timeline['visual_count']} caption, {timeline['speech_count']} câu nói, "
          f"~{clock(timeline['duration_hint'])}")

## Tải model lên 2 GPU

Mỗi GPU một bản `Qwen/Qwen3.5-4B` riêng (fp16/bf16 ~8GB, vừa T4 16GB) — **không** dùng `device_map='auto'`
vì cách đó xẻ một model qua cả hai GPU rồi chạy tuần tự, chậm hơn là hai worker độc lập.
Chỉ có 1 GPU thì notebook tự chạy single-GPU.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), (
    'Chưa bật GPU: Kaggle → Settings → Accelerator → GPU T4 x2'
    if ENV == 'kaggle' else 'Chưa bật GPU: Runtime → Change runtime type → GPU'
)

if DTYPE == 'auto':
    # T4 không có bfloat16 -> float16. A100/L4 thì bfloat16 an toàn hơn với activation lớn.
    model_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    model_dtype = {'float16': torch.float16, 'bfloat16': torch.bfloat16}[DTYPE]

GPU_COUNT = min(2, torch.cuda.device_count())
print(f'Phát hiện {torch.cuda.device_count()} GPU; sẽ dùng {GPU_COUNT} | dtype: {model_dtype}')
for gpu_id in range(GPU_COUNT):
    print(f'  cuda:{gpu_id}: {torch.cuda.get_device_name(gpu_id)}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# Batched generation cần padding bên trái, nếu không output sẽ lệch.
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_model(gpu_id):
    print(f'[GPU {gpu_id}] Đang tải {MODEL_ID}...')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=model_dtype, low_cpu_mem_usage=True, attn_implementation='sdpa',
        trust_remote_code=True,
    ).to(f'cuda:{gpu_id}')
    model.eval()
    print(f'[GPU {gpu_id}] Đã tải model')
    return model

models = [build_model(gpu_id) for gpu_id in range(GPU_COUNT)]
RUN_ID = f'{MODEL_ID}|summary-en|think={int(bool(ENABLE_THINKING))}|v1'
print('RUN_ID:', RUN_ID)

## Sinh summary

`chunk_events` cắt timeline theo **số token thật** (đếm bằng tokenizer) chứ không theo số dòng — video
nhiều thoại và video nhiều cảnh có mật độ token rất khác nhau. Một chunk thì gọi model đúng một lần với
`FINAL_PROMPT`; nhiều chunk thì map từng chunk theo batch rồi reduce các digest lại.

`generate_batch` gặp OOM sẽ tự lùi về sinh từng prompt một, nên `MAP_BATCH_SIZE` đặt hơi cao cũng không
làm chết cả lượt chạy.

In [ ]:
import traceback

def count_tokens(text):
    return len(tokenizer(text, add_special_tokens=False).input_ids)

def chunk_events(events, budget=None):
    """Cắt danh sách sự kiện thành các chunk vừa `budget` token."""
    budget = budget or CHUNK_TOKEN_BUDGET
    chunks, current, current_tokens = [], [], 0
    for event in events:
        line = event_line(event)
        tokens = count_tokens(line) + 1
        if current and current_tokens + tokens > budget:
            chunks.append(current)
            current, current_tokens = [], 0
        # Một dòng đơn lẻ dài hơn cả budget thì vẫn phải giữ: cắt text cho vừa.
        if tokens > budget:
            ids = tokenizer(line, add_special_tokens=False).input_ids[:budget]
            line = tokenizer.decode(ids)
            tokens = budget
        current.append(line)
        current_tokens += tokens
    if current:
        chunks.append(current)
    if MAX_CHUNKS_PER_VIDEO:
        chunks = chunks[:MAX_CHUNKS_PER_VIDEO]
    return chunks

def render_prompt(user_text):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user_text}]
    kwargs = {'tokenize': False, 'add_generation_prompt': True}
    try:
        return tokenizer.apply_chat_template(messages, enable_thinking=ENABLE_THINKING, **kwargs)
    except TypeError:
        # transformers cũ chưa có enable_thinking trong chat template.
        return tokenizer.apply_chat_template(messages, **kwargs)

THINK_BLOCK = re.compile(r'<think>.*?</think>', re.DOTALL)

def clean_output(text):
    text = THINK_BLOCK.sub('', text)
    text = text.replace('<think>', '').replace('</think>', '')
    return text.strip()

def generate_batch(model, prompts, max_new_tokens):
    """Sinh text cho nhiều prompt trên một GPU; OOM thì lùi về từng prompt một."""
    if not prompts:
        return []
    device = next(model.parameters()).device
    try:
        batch = tokenizer(prompts, return_tensors='pt', padding=True, add_special_tokens=False).to(device)
        torch.manual_seed(SEED)
        with torch.inference_mode():
            output = model.generate(
                **batch,
                max_new_tokens=max_new_tokens,
                do_sample=DO_SAMPLE,
                temperature=TEMPERATURE if DO_SAMPLE else None,
                top_p=TOP_P if DO_SAMPLE else None,
                top_k=TOP_K if DO_SAMPLE else None,
                pad_token_id=tokenizer.pad_token_id,
            )
        generated = output[:, batch.input_ids.shape[1]:]
        return [clean_output(text) for text in tokenizer.batch_decode(generated, skip_special_tokens=True)]
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(prompts) == 1:
            raise
        print(f'  OOM ở batch {len(prompts)} -> lùi về từng prompt một')
        results = []
        for prompt in prompts:
            results.extend(generate_batch(model, [prompt], max_new_tokens))
        return results

SECTION_PATTERN = re.compile(
    r'^\s*(SUMMARY|TOPICS|ENTITIES)\s*[:\-]\s*(.*?)(?=^\s*(?:SUMMARY|TOPICS|ENTITIES)\s*[:\-]|\Z)',
    re.IGNORECASE | re.DOTALL | re.MULTILINE,
)

def split_list(text):
    parts = [part.strip(' .;-') for part in re.split(r'[;\n]|(?:^|\s)[-*]\s', text) if part.strip(' .;-')]
    if len(parts) <= 1 and ',' in text:
        parts = [part.strip() for part in text.split(',') if part.strip()]
    return [part for part in parts if part.lower() not in {'none', 'n/a'}]

def parse_final(text):
    """Tách 3 nhãn; model trả về dạng khác thì dồn tất cả vào summary để không mất dữ liệu."""
    sections = {name.upper(): body.strip() for name, body in SECTION_PATTERN.findall(text)}
    summary = sections.get('SUMMARY', '').strip()
    if not summary:
        summary = re.sub(r'^\s*(TOPICS|ENTITIES)\s*[:\-].*$', '', text, flags=re.IGNORECASE | re.MULTILINE).strip()
    return {
        'summary': ' '.join(summary.split()),
        'topics': split_list(sections.get('TOPICS', '')),
        'entities': split_list(sections.get('ENTITIES', '')),
    }

def summarize_timeline(model, timeline, log=print):
    video_id, events = timeline['video_id'], timeline['events']
    chunks = chunk_events(events)
    chunk_summaries = []

    if len(chunks) > 1:
        prompts, spans = [], []
        for index, lines in enumerate(chunks, 1):
            start, end = lines[0].split(']')[0].lstrip('['), lines[-1].split(']')[0].lstrip('[')
            spans.append((start, end))
            prompts.append(render_prompt(MAP_PROMPT.format(
                video_id=video_id, part=index, total=len(chunks),
                start=start, end=end, evidence='\n'.join(lines),
            )))
        log(f'  map: {len(chunks)} chunk')
        for offset in range(0, len(prompts), MAP_BATCH_SIZE):
            chunk_summaries.extend(generate_batch(model, prompts[offset:offset + MAP_BATCH_SIZE], MAX_NEW_TOKENS_MAP))
        evidence = '\n\n'.join(
            f'--- part {index}/{len(chunks)} ({spans[index - 1][0]}-{spans[index - 1][1]}) ---\n{summary}'
            for index, summary in enumerate(chunk_summaries, 1)
        )
    else:
        evidence = '\n'.join(chunks[0]) if chunks else ''

    final_text = generate_batch(
        model, [render_prompt(FINAL_PROMPT.format(video_id=video_id, evidence=evidence))], MAX_NEW_TOKENS_FINAL,
    )[0]
    parsed = parse_final(final_text)
    return {
        'video_id': video_id,
        'model': MODEL_ID,
        'run_id': RUN_ID,
        'language': 'en',
        'enable_thinking': bool(ENABLE_THINKING),
        'summary': parsed['summary'],
        'topics': parsed['topics'],
        'entities': parsed['entities'],
        'raw_output': final_text,
        'chunk_summaries': chunk_summaries,
        'num_chunks': len(chunks),
        'evidence': {
            'visual_count': timeline['visual_count'],
            'speech_count': timeline['speech_count'],
            'has_caption': timeline['has_caption'],
            'has_transcript': timeline['has_transcript'],
            'duration_hint': round(timeline['duration_hint'], 2),
            'caption_source': str(CAPTION_FILES.get(video_id, '')),
            'transcript_source': str(TRANSCRIPT_FILES.get(video_id, '')),
        },
        'complete': True,
    }

def output_json_path(video_id):
    return OUTPUT_ROOT / f'{video_id}.json'

def atomic_write(path, text):
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(text, encoding='utf-8')
    temp.replace(path)

def save_summary(payload):
    video_id = payload['video_id']
    atomic_write(output_json_path(video_id), json.dumps(payload, ensure_ascii=False, indent=2))
    lines = [payload['summary'], '']
    if payload['topics']:
        lines.append('TOPICS: ' + '; '.join(payload['topics']))
    if payload['entities']:
        lines.append('ENTITIES: ' + '; '.join(payload['entities']))
    atomic_write(OUTPUT_ROOT / f'{video_id}.txt', '\n'.join(lines) + '\n')

## Resume — nhận kết quả của các session trước

`/kaggle/working` bị xoá khi session kết thúc, nên phải mang kết quả cũ quay lại:

1. Lần chạy trước bấm **Save Version**.
2. Lần chạy mới: **Add Data → Your Work / Notebook Output**, chọn output đó.
3. Chạy lại từ đầu — cell này chép các `.json` cũ về `OUTPUT_ROOT`, những video đó sẽ in `SKIP`.

File `summaries.zip` do cell cuối tạo cũng được tự giải nén ở đây.

In [ ]:
import shutil
import zipfile

RESUME_ROOTS = []
if ENV == 'kaggle' and Path('/kaggle/input').is_dir():
    source_branch = {parent for directory in CAPTION_DIRS + TRANSCRIPT_DIRS
                     for parent in (directory, *directory.parents)}
    RESUME_ROOTS = [p for p in sorted(Path('/kaggle/input').iterdir())
                    if p.is_dir() and p.resolve() not in source_branch]

restored = 0
for resume_root in RESUME_ROOTS:
    for zip_path in sorted(resume_root.rglob('summaries*.zip')):
        try:
            with zipfile.ZipFile(zip_path) as archive:
                members = [name for name in archive.namelist() if name.endswith(('.json', '.txt'))]
                archive.extractall(OUTPUT_ROOT, members=members)
            print(f'Đã giải nén {len(members)} file từ {zip_path}')
        except zipfile.BadZipFile:
            print('Bỏ qua (zip hỏng):', zip_path)
    for json_path in resume_root.rglob('*.json'):
        if not VIDEO_ID_PATTERN.match(json_path.stem):
            continue
        payload = read_json(json_path)
        if not payload or payload.get('run_id') != RUN_ID or not payload.get('complete'):
            continue   # file của model/prompt khác (hoặc caption/transcript) -> không phải summary của mình
        destination = output_json_path(json_path.stem)
        if destination.exists() or json_path.resolve() == destination.resolve():
            continue
        shutil.copy2(json_path, destination)
        sidecar = json_path.with_suffix('.txt')
        if sidecar.exists():
            shutil.copy2(sidecar, OUTPUT_ROOT / sidecar.name)
        restored += 1

print('Resume — quét ở:', [str(p) for p in RESUME_ROOTS] or '(không có)')
print(f'Chép về {restored} summary của lần chạy trước')

def is_done(video_id):
    path = output_json_path(video_id)
    if not path.is_file():
        return False
    payload = read_json(path)
    return bool(payload and payload.get('run_id') == RUN_ID and payload.get('complete') and payload.get('summary'))

def parse_slice(text):
    text = (text or '').strip()
    if not text:
        return None
    start, separator, stop = text.partition(':')
    if not separator:
        return slice(int(start), int(start) + 1)
    return slice(int(start) if start.strip() else None, int(stop) if stop.strip() else None)

done = [video_id for video_id in video_ids if is_done(video_id)]
pending = [video_id for video_id in video_ids if video_id not in set(done)]
if not OVERWRITE:
    selection = parse_slice(RUN_SLICE)
    pending = pending[selection] if selection else pending
else:
    pending = video_ids[parse_slice(RUN_SLICE)] if parse_slice(RUN_SLICE) else list(video_ids)

print(f'\n{len(selected_folders)} folder'
      + (f' | prefix={VIDEO_PREFIXES}' if VIDEO_PREFIXES else '')
      + f': {len(video_ids)} video | đã xong {len(done)} | lượt này {len(pending)}')
print('Sẽ chạy:', ', '.join(pending[:12]) + (' ...' if len(pending) > 12 else '') or '(không có)')

## Chạy thử một video

Chạy cell này trước để xác nhận evidence, prompt và chất lượng summary. Cell **có ghi file** cho video đó
(nên nó sẽ bị `SKIP` ở lượt chạy đầy đủ) và in ra thời gian để ước lượng cả đợt.

In [ ]:
import time

assert video_ids, 'Không có video nào để chạy'
sample_id = pending[0] if pending else video_ids[0]
sample_timeline = timelines[sample_id]

print(f"{sample_id}: {sample_timeline['visual_count']} caption + {sample_timeline['speech_count']} câu nói, "
      f"~{clock(sample_timeline['duration_hint'])}, {len(chunk_events(sample_timeline['events']))} chunk")
print('\n--- 8 DÒNG EVIDENCE ĐẦU ---')
for event in sample_timeline['events'][:8]:
    print(event_line(event)[:200])

started = time.time()
sample_payload = summarize_timeline(models[0], sample_timeline)
elapsed = time.time() - started
save_summary(sample_payload)

print(f'\n--- SUMMARY ({elapsed:.1f}s) ---\n{sample_payload["summary"]}')
print('\nTOPICS:', '; '.join(sample_payload['topics']) or '(rỗng)')
print('ENTITIES:', '; '.join(sample_payload['entities']) or '(rỗng)')
if not sample_payload['summary']:
    print('\nCẢNH BÁO: summary rỗng — xem raw_output:\n', sample_payload['raw_output'][:1500])
print(f'\nĐã lưu: {output_json_path(sample_id)}')
print(f'Ước lượng: {elapsed:.0f}s/video / GPU -> {len(pending)} video với {GPU_COUNT} GPU '
      f'≈ {elapsed * len(pending) / max(1, GPU_COUNT) / 60:.0f} phút')

## Chạy toàn bộ trên 2 GPU

Hai worker rút video từ một queue chung, mỗi worker dùng bản model của GPU mình. Video đã có `.json` khớp
`RUN_ID` được bỏ qua, nên **chạy lại cell này là tiếp tục từ chỗ dừng**. Video lỗi được ghi vào `_failed.json`
và không làm dừng cả lượt.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from queue import Empty, Queue
from threading import Lock

jobs = Queue()
for index, video_id in enumerate(pending, 1):
    jobs.put((index, video_id))
total = len(pending)
print_lock = Lock()

def gpu_worker(gpu_id):
    model = models[gpu_id]
    success = skipped = failed = 0
    failures = []
    with torch.cuda.device(gpu_id):
        while True:
            try:
                index, video_id = jobs.get_nowait()
            except Empty:
                break
            try:
                if not OVERWRITE and is_done(video_id):
                    skipped += 1
                    with print_lock:
                        print(f'[GPU {gpu_id}] [{index}/{total}] SKIP {video_id}')
                    continue
                with print_lock:
                    print(f'[GPU {gpu_id}] [{index}/{total}] SUM  {video_id}')
                started = time.time()
                payload = summarize_timeline(
                    model, timelines[video_id],
                    log=lambda message: None,   # log của map giữ im để output không rối khi 2 GPU chạy song song
                )
                save_summary(payload)
                success += 1
                with print_lock:
                    print(f'[GPU {gpu_id}] [{index}/{total}] OK   {video_id} '
                          f'({time.time() - started:.0f}s, {payload["num_chunks"]} chunk, '
                          f'{len(payload["summary"].split())} từ)')
            except Exception as error:
                failed += 1
                failures.append({'video_id': video_id, 'gpu': gpu_id, 'error': repr(error)})
                with print_lock:
                    print(f'[GPU {gpu_id}] ERROR {video_id}: {error!r}')
                    print(traceback.format_exc())
            finally:
                jobs.task_done()
                torch.cuda.empty_cache()
    return success, skipped, failed, failures

with ThreadPoolExecutor(max_workers=GPU_COUNT) as executor:
    worker_results = list(executor.map(gpu_worker, range(GPU_COUNT)))

success = sum(item[0] for item in worker_results)
skipped = sum(item[1] for item in worker_results)
failed = sum(item[2] for item in worker_results)
failures = [failure for item in worker_results for failure in item[3]]
atomic_write(OUTPUT_ROOT / '_failed.json', json.dumps(failures, ensure_ascii=False, indent=2))
print(f'\nHoàn tất với {GPU_COUNT} GPU: success={success}, skipped={skipped}, failed={failed}, total={total}')
print('Output:', OUTPUT_ROOT)

## Kiểm tra nhanh + đóng gói

In vài summary vừa sinh để soi bằng mắt, rồi nén `Summary_video/` thành `summaries.zip` trong
`/kaggle/working` để tải về ở tab **Output** (hoặc Save Version để dùng resume cho lần sau).
Trên Colab kết quả đã nằm sẵn trên Drive nên cell chỉ báo bỏ qua.

In [ ]:
import textwrap

files = sorted(OUTPUT_ROOT.glob('L*.json'))
print(f'{len(files)} summary trong {OUTPUT_ROOT}\n')
for path in files[:3]:
    payload = read_json(path) or {}
    print(f"=== {payload.get('video_id')} ({payload.get('num_chunks')} chunk) ===")
    print(textwrap.fill(payload.get('summary', ''), 100))
    print('TOPICS:', '; '.join(payload.get('topics') or []))
    print('ENTITIES:', '; '.join(payload.get('entities') or []), '\n')

empty = [path.stem for path in files if not (read_json(path) or {}).get('summary')]
if empty:
    print('CẢNH BÁO: summary rỗng ở', ', '.join(empty[:20]))

if ENV == 'kaggle':
    archive = shutil.make_archive('/kaggle/working/summaries', 'zip', root_dir=OUTPUT_ROOT)
    print(f'Đã đóng gói: {archive} ({Path(archive).stat().st_size / 1024 / 1024:.1f} MB)')
    print('Tải về ở panel Output bên phải, hoặc Save Version để giữ lại cho lần resume sau.')
else:
    print('Bỏ qua đóng gói: kết quả đã nằm ở', OUTPUT_ROOT)